# Derived vs catalogued amplitude: a sample of classical Cepheids
Question: for a sample of classical Cepheids, do V amplitudes derived from AAVSO observations agree with the catalogued VSX V amplitude, and by how much do they differ?

## Selection rules (written before looking at any results)
- Variability type DCEP in VSX, with a V-band magnitude range and a period listed.
- Period between 3 and 15 days (so 2024-2025 covers at least about 50 cycles).
- Bright enough to be well observed: brighter than V = 8 at maximum (threshold may change after seeing observation counts, but not after seeing amplitude results).
- Observations: Johnson V, 2024-01-01 to 2025-12-31, same download settings as del Cep.
- Keep a star only if it has at least N observations (N to be fixed after seeing counts).
- The same pipeline (check-star screening, order-3 Fourier fit, bootstrap) is applied to every star.
- Scope: conclusions apply only to bright, well-observed classical Cepheids.

In [1]:
   import pandas as pd
   v = pd.read_csv("../data/raw/vsx_dcep_search_20261014.csv")
   print(v.shape)
   print(v.columns.tolist())
   print(v.head(5).to_string())

(57, 8)
['Name', 'AUID', 'Coords', 'Const', 'Type', 'Period', 'Mag', 'Observations']
                      Name         AUID                   Coords Const  Type     Period               Mag  Observations
0                    U Aql  000-BCT-642  19 29 21.36 -07 02 38.7   Aql  DCEP   7.024110     6.08 - 6.86 V          6281
1                   TT Aql  000-BCF-419  19 08 13.75 +01 17 55.2   Aql  DCEP  13.754900     6.53 - 7.65 V         12136
2                  eta Aql  000-BCT-763  19 52 28.37 +01 00 20.4   Aql  DCEP   7.176790     3.49 - 4.30 V         36424
3  ZTF J192152.00+150346.9  000-BPQ-419  19 21 52.01 +15 03 47.0   Aql  DCEP   8.777444  16.582 (0.500) g           471
4                   RT Aur  000-BBL-373  06 28 34.09 +30 29 34.9   Aur  DCEP   3.728240     5.00 - 5.82 V         13348


In [2]:
import re
import numpy as np
import pandas as pd

def parse_mag(s):
    """Parse VSX Mag strings like '6.08 - 6.86 V'. Returns (bright, faint, band) or NaN."""
    m = re.match(r"^\s*([\d.]+)\s*-\s*([\d.]+)\s+([A-Za-z]+)\s*$", str(s))
    if m:
        return float(m.group(1)), float(m.group(2)), m.group(3)
    return np.nan, np.nan, None

v[["mag_bright", "mag_faint", "band"]] = v["Mag"].apply(lambda s: pd.Series(parse_mag(s)))
v["vsx_amp"] = v["mag_faint"] - v["mag_bright"]

# 1. parsing results
print("band counts:")
print(v["band"].value_counts(dropna=False))
print()
print("rows that are not V-band or did not parse:")
print(v[v["band"] != "V"][["Name", "Mag", "band"]].to_string())
print()
print("variability types:")
print(v["Type"].value_counts())

# 2. do the period and brightness cuts hold?
print()
print("period summary:")
print(v["Period"].describe().round(3))
print()
vv = v[v["band"] == "V"]
print(len(vv), "rows with a V-band range")
print(vv["mag_bright"].describe().round(2))

# 3. del Cep, to check the hand-typed values from notebook 05
print()
print("del Cep rows:")
print(v[v["Name"].str.contains("del Cep", case=False)].to_string())

band counts:
band
V      42
NaN    15
Name: count, dtype: int64

rows that are not V-band or did not parse:
                       Name                Mag band
3   ZTF J192152.00+150346.9   16.582 (0.500) g  NaN
15                   TY Cru  11.021 (0.380) Ic  NaN
17                V2525 Cyg   12.983 (0.573) g  NaN
18    WISE J195222.3+421516  10.981 (0.153) W1  NaN
20    WISE J054003.9-672352  13.472 (0.165) W1  NaN
23                LMC V0792     14.28 (1.35) V  NaN
24                   AA Mon   13.432 (0.868) g  NaN
25                   FI Mon   12.473 (0.544) r  NaN
30                   CV Ori  11.734 (0.575) Ic  NaN
33             NSVS 1951077   12.755 (0.512) g  NaN
35                   CM Pup   13.528 (0.609) r  NaN
36                   HW Pup  10.508 (0.617) Ic  NaN
38  ZTF J193156.28+183312.4   16.195 (0.745) g  NaN
47         OGLE-BLG-CEP-076  13.833 (0.575) Ic  NaN
48         [DMM2015] VVV-17    12.13 (0.25) Ks  NaN

variability types:
Type
DCEP    57
Name: count, dtype: int6

In [3]:
sel = v[v["band"] == "V"].copy()
print("V-band range rows:", len(sel))

sel = sel[(sel["Period"] >= 3) & (sel["Period"] <= 15)]
print("after period 3-15 d:", len(sel))

sel = sel[sel["mag_bright"] < 8]
print("after V < 8 at maximum:", len(sel))

print()
print(v["Type"].value_counts().to_string())
print()
print(sel[["Name", "AUID", "Period", "mag_bright", "mag_faint", "vsx_amp", "Observations"]]
      .sort_values("Observations", ascending=False).to_string())

V-band range rows: 42
after period 3-15 d: 42
after V < 8 at maximum: 42

Type
DCEP    57

         Name         AUID     Period  mag_bright  mag_faint  vsx_amp  Observations
10    del Cep  000-BCQ-471   5.366290        3.49       4.36     0.87         70654
2     eta Aql  000-BCT-763   7.176790        3.49       4.30     0.81         36424
22    zet Gem  000-BBM-362  10.149806        3.62       4.18     0.56         23925
37      S Sge  000-BCW-946   8.382086        5.24       6.04     0.80         16386
55      T Vul  000-BCM-658   4.435380        5.41       6.09     0.68         14369
4      RT Aur  000-BBL-373   3.728240        5.00       5.82     0.82         13348
1      TT Aql  000-BCF-419  13.754900        6.53       7.65     1.12         12136
16     SU Cyg  000-BCV-605   3.845570        6.44       7.22     0.78         11064
26      R Mus  000-BCW-450   7.510400        5.93       6.73     0.80          9854
19    bet Dor  000-BCV-841   9.842675        3.41       4.08     0.67